In [ ]:
!pip install roboflow ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 9.0 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2


In [ ]:
# train.py
# Run this in Google Colab (Runtime > Change runtime type > GPU).
#
# Step 1: Install dependencies (Colab: run as a cell with a leading !)
#   pip install ultralytics roboflow
#
# IMPORTANT: This downloads the dataset directly from Roboflow's servers into
# your Colab session. Nothing is uploaded from or processed on your own
# computer -- your laptop just needs a browser tab open. All the heavy work
# (download, unzip, training) happens on Google's cloud hardware.

import os
from roboflow import Roboflow
from ultralytics import YOLO
from google.colab import drive

# -----------------------------------------------------------------------
# STEP 0: Mount Google Drive so we can safely save results at the end
# -----------------------------------------------------------------------
drive.mount('/content/drive')

# -----------------------------------------------------------------------
# STEP 1: Download the dataset directly into Colab (not through your PC)
# -----------------------------------------------------------------------
rf = Roboflow(api_key="IN9sfHttwwoh80KDbOF2")  # roboflow.com -> Settings -> API Key
project = rf.workspace("lexiwave-glqpu").project("construction-worker-9yqmx")
version = project.version(1)
dataset = version.download("yolov8")

DATA_ROOT = dataset.location  # e.g. "construction-worker-9yqmx-1"

# -----------------------------------------------------------------------
# STEP 2: Remove the 'boots' class from labels and re-index the rest
# -----------------------------------------------------------------------
# Original order (alphabetical, per data.yaml): 0=boots, 1=gloves, 2=helmet, 3=human, 4=vest
# New order after dropping boots: 0=gloves, 1=helmet, 2=human, 3=vest
BOOTS_INDEX = 0
REMAP = {1: 0, 2: 1, 3: 2, 4: 3}  # old_index -> new_index (boots/0 is dropped)

def clean_labels(split):
    label_dir = os.path.join(DATA_ROOT, split, "labels")
    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        path = os.path.join(label_dir, fname)
        new_lines = []
        with open(path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                old_cls = int(parts[0])
                if old_cls == BOOTS_INDEX:
                    continue  # drop boots annotations entirely
                new_cls = REMAP[old_cls]
                new_lines.append(" ".join([str(new_cls)] + parts[1:]))
        with open(path, "w") as f:
            f.write("\n".join(new_lines) + ("\n" if new_lines else ""))

for split in ["train", "valid", "test"]:
    clean_labels(split)

# Write a corrected data.yaml with boots removed
new_yaml = f"""train: {DATA_ROOT}/train/images
val: {DATA_ROOT}/valid/images
test: {DATA_ROOT}/test/images

nc: 4
names: ['gloves', 'helmet', 'human', 'vest']
"""
yaml_path = os.path.join(DATA_ROOT, "data_no_boots.yaml")
with open(yaml_path, "w") as f:
    f.write(new_yaml)

print(f"Cleaned dataset ready. Using config: {yaml_path}")

# -----------------------------------------------------------------------
# STEP 3: Train
# -----------------------------------------------------------------------
model = YOLO("yolov8n.pt")  # nano model: fastest to train, good baseline
# swap to "yolov8s.pt" for a bit more accuracy if training time allows

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,       # early stopping if val loss plateaus
    name="ppe_yolov8n",
    project="runs/ppe"
)

# -----------------------------------------------------------------------
# STEP 4: Evaluate
# -----------------------------------------------------------------------
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("Per-class mAP50:", metrics.box.maps)  # array aligned to class indices in data.yaml

# Confusion matrix and PR curves are auto-saved to runs/ppe/ppe_yolov8n/
# Look for confusion_matrix.png and results.png after training completes.

# -----------------------------------------------------------------------
# STEP 5: Save your results to Google Drive (survives session disconnects)
# -----------------------------------------------------------------------
import shutil
import glob

best_pt_candidates = glob.glob("runs/**/best.pt", recursive=True)
if not best_pt_candidates:
    raise FileNotFoundError("Could not find best.pt -- check the runs/ folder structure printed above.")

best_pt_path = best_pt_candidates[0]
print("Found weights at:", best_pt_path)

DRIVE_SAVE_DIR = "/content/drive/MyDrive/ppe-detection-results"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

shutil.copy(best_pt_path, os.path.join(DRIVE_SAVE_DIR, "best.pt"))
confusion_matrix_path = best_pt_path.replace("weights/best.pt", "confusion_matrix.png")
results_png_path = best_pt_path.replace("weights/best.pt", "results.png")
if os.path.exists(confusion_matrix_path):
    shutil.copy(confusion_matrix_path, os.path.join(DRIVE_SAVE_DIR, "confusion_matrix.png"))
if os.path.exists(results_png_path):
    shutil.copy(results_png_path, os.path.join(DRIVE_SAVE_DIR, "results.png"))

print(f"Saved best.pt and evaluation images to your Google Drive at: {DRIVE_SAVE_DIR}")
print("Open drive.google.com on your computer and download from the 'ppe-detection-results' folder --")
print("this copy is safe even if this Colab session disconnects.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to construction-worker-1 in yolov8:: 100%|██████████| 10008/10008 [00:01<00:00, 6372.74it/s]


Cleaned dataset ready. Using config: /content/construction-worker-1/data_no_boots.yaml
Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/construction-worker-1/data_no_boots.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yo

In [ ]:
python3 infer_webcam.py

SyntaxError: invalid syntax (3997975889.py, line 1)